# Scientific Machine Learning HW 2 - PINN
Daniel Sun

In [8]:
import torch
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")
print(f"Using {device} device")
try:
    print(torch.cuda.get_device_name(device))
except:
    print("CUDA unavailable")
from torch import nn

from tqdm import tqdm
import numpy as np
from matplotlib import pyplot as plt
from scipy.integrate import solve_ivp

import os
if os.name == 'posix':
    # Cluster
    os.chdir("/home/dsun/PSI-Machine-Learning/Homework 2")
print(os.getcwd())

from pinn import data, FCNN, PINN

Using cuda device
NVIDIA GeForce GTX 1660 Ti with Max-Q Design
c:\Users\danie\Documents\GitHub\PSI-Machine-Learning\Homework 2


In [2]:
DEV = True # Less epochs for faster development

In [3]:
# Lotka-Volterra gradient
def grad(_, p, LV_params):
    x, y = p
    alpha, beta, gamma, delta = LV_params
    return np.array([x * (alpha - beta * y), y * (delta * x - gamma)])

# Jacobian for Radau solver
def jac(_, p, LV_params):
    x, y = p
    alpha, beta, gamma, delta = LV_params
    return np.array([[alpha - beta * y, -beta * x], [delta * y, delta * x - gamma]])

In [4]:
data_dict = {}

In [5]:
params = np.array([0.6, 0.025, 0.5, 0.0125])
t0, tf = (0, 24) # Period is ~11.47
t = np.arange(t0, tf + 1)
p0 = np.array([30, 4]) # Initial condition [hare, lynx]

p_exact = solve_ivp(grad, (t0, tf), p0, t_eval = t, jac = jac, args = (params,), method = "Radau", dense_output = True).y.T
data_dict["synthetic"] = data(p_exact, "synthetic", "Synthetic Data", 12)

In [6]:
def train(train_dataloader, model, loss_fn, optimizer):
    model.train()
    train_loss = 0
    for t, p in train_dataloader:
        pred = model(t)
        loss = loss_fn(pred, p)
    
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        train_loss += loss.item()
    train_loss /= len(train_dataloader)
    return train_loss

def eval(eval_dataloader, model, loss_fn):
    model.eval()
    eval_loss = 0
    with torch.no_grad():
        for t, p in eval_dataloader:
            pred = model(t)
            loss = loss_fn(pred, p)
            eval_loss += loss.item()
    eval_loss /= len(eval_dataloader)
    return eval_loss

In [7]:
def plot_loss(train_loss, eval_loss, title, filename):
    plt.figure(dpi = 300, layout = "constrained")
    plt.plot(train_loss, label = "Train Loss")
    plt.plot(eval_loss, label = "Eval Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.savefig(f"plots/{filename}.pdf")
    plt.show()

### PINN

In [9]:
def pinn_loss(pred, p, d):
    mse_loss = nn.MSELoss()(pred, p)

    T = 2 * d.period * d.dt / d.t_scale # 2 periods, 100 collocation points per period
    t_physics = torch.linspace(0, T, 200, device = device, requires_grad = True).unsqueeze(1)
    pred_physics = d.pinn(t_physics)
    pred_hare = pred_physics[:, 0].unsqueeze(1)
    pred_lynx = pred_physics[:, 1].unsqueeze(1)
    dhare_physics = torch.autograd.grad(pred_hare, t_physics, torch.ones_like(pred_hare), create_graph = True)[0]
    dlynx_physics = torch.autograd.grad(pred_lynx, t_physics, torch.ones_like(pred_lynx), create_graph = True)[0]
    dp_physics = torch.cat([dhare_physics, dlynx_physics], dim = 1)
    
    dp_target = torch.zeros_like(dp_physics)
    dp_target[:, 0] = d.pinn.alpha * pred_physics[:, 0] - d.pinn.beta * pred_physics[:, 0] * pred_physics[:, 1]
    dp_target[:, 1] = -d.pinn.gamma * pred_physics[:, 1] + d.pinn.delta * pred_physics[:, 0] * pred_physics[:, 1]
    physics_loss = nn.MSELoss()(dp_physics, dp_target)

    # Scale physics_loss ~ 1e-4
    return mse_loss, physics_loss * (d.period * d.dt / 2 / d.t_scale)**2 * 0.01

In [10]:
def train_pinn(d, epochs):
    loss_fn = lambda pred, p: sum(pinn_loss(pred, p, d))
    loss_fn_fast = nn.MSELoss()
    optimizer = torch.optim.Adam(d.pinn.parameters(), lr = 3e-4)

    train_loss = np.zeros(epochs)
    eval_loss = np.zeros(epochs)

    for i in tqdm(range(epochs)):
        train_loss[i] = train(d.train_dataloader, d.pinn, loss_fn, optimizer)
        eval_loss[i] = eval(d.dataloader, d.pinn, loss_fn_fast)
        if (i + 1) % (epochs // 10) == 0:
            print(f"Epoch {i + 1}/{epochs}, Train Loss: {train_loss[i]:.4f}, Eval Loss: {eval_loss[i]:.4f}")
            d.pinn.eval()
            with torch.no_grad():
                pred = d.pinn(d.t_norm)
            curr_loss = pinn_loss(pred, d.p_norm, d)
            print(f"MSE Loss: {curr_loss[0]:.4f}, Physics Loss: {curr_loss[1]:.4f}")
            d.plot_pred(pred, f"{d.longname}, PINN Prediction, Epoch {i + 1}", figsize = (20, 5))

            w = d.unnormalize_w(torch.tensor([d.pinn.alpha.detach(), d.pinn.beta.detach(), d.pinn.gamma.detach(), d.pinn.delta.detach()]))
            if d.name in ["synthetic", "synthetic_noisy"]:
                print("Exact parameters: ", params)
            print("PINN learned parameters: ", w)
            if d.name in ["synthetic", "synthetic_noisy"]:
                print("Relative error: ", (w - params) / np.abs(params))
            print()
    
    return train_loss, eval_loss

In [ ]:
epochs = 1000 if DEV else 60000 # Roughly 1 hour per 20000 epochs per dataset

for d in data_dict.values():
    try:
        d.pinn = torch.load(f"models/{d.name}_pinn.pt", weights_only = False)
        print(f"Loaded trained PINN for {d.name}")
    except FileNotFoundError:
        d.pinn = PINN(params / d.w_scale).to(device)
        # Transfer weights from FCNN
        d.fcnn = torch.load(f"models/{d.name}_fcnn.pt", weights_only = False)
        d.pinn.load_state_dict(d.fcnn.state_dict(), strict = False)
        train_loss, eval_loss = train_pinn(d, epochs)
        plot_loss(train_loss, eval_loss, f"{d.longname}, PINN Loss", f"{d.name}_pinn_loss")
        if not DEV:
            torch.save(d.pinn, f"models/{d.name}_pinn.pt")
    
    if DEV:
        break

  4%|▍         | 40/1000 [00:18<07:42,  2.07it/s]

In [ ]:
for d in data_dict.values():
    d.pinn.eval()
    pred = d.pinn(d.t_norm)
    d.plot_pred(pred, f"{d.longname}, PINN Prediction", f"{d.name}_pinn_prediction")

    w = d.unnormalize_w(torch.tensor([d.pinn.alpha.detach(), d.pinn.beta.detach(), d.pinn.gamma.detach(), d.pinn.delta.detach()]))
    if d.name in ["synthetic", "synthetic_noisy"]:
        print("Exact parameters: ", params)
    print("PINN learned parameters: ", w)
    if d.name in ["synthetic", "synthetic_noisy"]:
        print("Relative error: ", (w - params) / np.abs(params))

    if DEV:
        break